In [7]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect, domain_best_by_model_with_baseline_delta
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature
from titanic_ml.common.data.eda import sample_dataframe


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["fe09__ticket_group_size"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     result_df = save_results(exp_result)
#     save_configs(exp_config)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe09__ticket_group_size
Experiment: fe10__fare_per_ticket_member
Experiment: fe11__age_bin
Experiment: fe12__sex_pclass
Experiment: cb01__age_and_bins
Experiment: cb02__age_imputed_title_and_bins
Experiment: cb03__age_imputed_title_Pclass_and_bins


In [9]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [10]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

running exp: {'name': 'fe09__ticket_group_size__logreg', 'features': ['Pclass', 'Sex', 'SibSp', 'Parch', 'Age', 'Fare', 'Embarked', 'TicketGroupSize'], 'feature_engineering': [<function add_ticket_group_size at 0x0000020781400EE0>], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare', 'TicketGroupSize'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 09: imputing age by grouped impute using title and pclass, expected to give better results then imputing with only title', 'stage': 'fe09', 'feature_group': 'ticket_group_size', 'group': 'fe09__ticket_group_size', 'domain': 'ticket'}
running exp: {'n

In [11]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()

# For combos:
all_results = load_results()
references = ['cb01__age_and_bins','cb02__age_imputed_title_and_bins']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
### fe09__ticket_group_size

_Description pending._

<details>
<summary>Conclusion</summary>


#### Interpretation

- Verdict: mixed
- Recommended for specific models:
  - svc: test_accuracy_mean: 0.005
    - Secondary gains:
      - test_f1_mean: 0.01


#### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group           | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:------------------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe09__ticket_group_size | logreg        |                          0.786 |                    

In [12]:
| reference_group   | compare_group           | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:------------------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe09__ticket_group_size | logreg        |                          0.786 |                        0.788 |                      0.002 |                    0.713 |                  0.715 |                0.002 |
| baseline__raw     | fe09__ticket_group_size | knn           |                          0.809 |                        0.804 |                     -0.005 |                    0.742 |                  0.733 |               -0.009 |
| baseline__raw     | fe09__ticket_group_size | svc           |                          0.827 |                        0.832 |                      0.005 |                    0.76  |                  0.77  |                0.01  |
| baseline__raw     | fe09__ticket_group_size | decision_tree |                          0.803 |                        0.8   |                     -0.003 |                    0.702 |                  0.702 |                0     |
| baseline__raw     | fe09__ticket_group_size | random_forest |                          0.822 |                        0.82  |                     -0.002 |                    0.744 |                  0.747 |                0.003 |
| baseline__raw     | fe09__ticket_group_size | extra_trees   |                          0.804 |                        0.806 |                      0.002 |                    0.721 |                  0.724 |                0.003 |
| baseline__raw     | fe09__ticket_group_size | xgb           |                          0.826 |                        0.818 |                     -0.008 |                    0.758 |                  0.749 |               -0.009 |

SyntaxError: invalid syntax (3250148173.py, line 1)

In [ ]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [ ]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [ ]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [ ]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [ ]:
# print(workflow["all_results"])

In [ ]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

In [ ]:
# bins = [0, 14, 35, 60, 100]
# labels = ['0', '2', '3', '1']
# exp_df['Age_bin'] = pd.cut(exp_df['Age'], bins=bins, labels=labels, right=False)

In [ ]:
# for age in range(0,18):
#     test_df = exp_df[exp_df['Age'] == age][['Age','Survived']]
#     test_df = test_df.groupby('Survived').value_counts()
#     print(test_df)

In [ ]:
age_best_delta = domain_best_by_model_with_baseline_delta(
    results_df=all_results,
    domain="age",
    metric="test_accuracy_mean",
)

age_best_delta

,model_name,experiment,group,domain,test_accuracy_mean,test_f1_mean,test_accuracy_mean_baseline,test_accuracy_mean_delta_vs_baseline
0,random_forest,fe11__age_bin__random_forest,fe11__age_bin,age,0.833,0.759,0.822,0.011
1,svc,cb02__age_imputed_title_and_bins__svc,cb02__age_imputed_title_and_bins,age,0.832,0.767,0.827,0.005
2,xgb,cb02__age_imputed_title_and_bins__xgb,cb02__age_imputed_title_and_bins,age,0.832,0.762,0.826,0.006
3,extra_trees,fe11__age_bin__extra_trees,fe11__age_bin,age,0.819,0.737,0.804,0.015
4,knn,fe11__age_bin__knn,fe11__age_bin,age,0.810,0.737,0.809,0.001
5,decision_tree,fe11__age_bin__decision_tree,fe11__age_bin,age,0.809,0.722,0.803,0.006
6,logreg,cb03__age_imputed_title_Pclass_and_bins__logreg,cb03__age_imputed_title_Pclass_and_bins,age,0.802,0.730,0.786,0.016


In [ ]:
eda = run_eda(train_df, target=TARGET, display=True, head=3,random=4, tail=3)
# print()

Dataframe Health
rows                                                                  891
columns                                                                12
duplicate_rows                                                          0
duplicate_%                                                           0.0
rows_with_missing                                                     708
rows_with_missing_%                                                 79.46
total_missing_values                                                  866
dataFrame_columns       [PassengerId, Survived, Pclass, Name, Sex, Age...
memory_usage                                Total memory usage: 315.03 KB
dtype: object
DataFrame Summary:
               dtype  non_null_count  missing_count  missing_%  unique  \
PassengerId    int64             891              0       0.00     891   
Survived       int64             891              0       0.00       2   
Pclass         int64             891              0       0.00

In [ ]:
print(eda)

{'dataFrame_summary':                dtype  non_null_count  missing_count  missing_%  unique  \
PassengerId    int64             891              0       0.00     891   
Survived       int64             891              0       0.00       2   
Pclass         int64             891              0       0.00       3   
Name          object             891              0       0.00     891   
Sex           object             891              0       0.00       2   
Age          float64             714            177      19.87      89   
SibSp          int64             891              0       0.00       7   
Parch          int64             891              0       0.00       7   
Ticket        object             891              0       0.00     681   
Fare         float64             891              0       0.00     248   
Cabin         object             204            687      77.10     148   
Embarked      object             889              2       0.22       4   

             ca

In [ ]:
print(eda['dataFrame_health'].to_markdown())

|                      | 0                                                                                                                    |
|:---------------------|:---------------------------------------------------------------------------------------------------------------------|
| rows                 | 891                                                                                                                  |
| columns              | 12                                                                                                                   |
| duplicate_rows       | 0                                                                                                                    |
| duplicate_%          | 0.0                                                                                                                  |
| rows_with_missing    | 708                                                                                                            

In [ ]:
print(eda["dataFrame_summary"].to_markdown())

|             | dtype   |   non_null_count |   missing_count |   missing_% |   unique |   cardinality_% | cardinality_label   | top_value           |   dominance_% | dominance_label   | bottom_value             |
|:------------|:--------|-----------------:|----------------:|------------:|---------:|----------------:|:--------------------|:--------------------|--------------:|:------------------|:-------------------------|
| PassengerId | int64   |              891 |               0 |        0    |      891 |          100    | potential_id        | 891                 |      0.112233 | balanced          | 12                       |
| Survived    | int64   |              891 |               0 |        0    |        2 |            0.22 | low_cardinality     | 0                   |     61.6162   | some_dominance    | 1                        |
| Pclass      | int64   |              891 |               0 |        0    |        3 |            0.34 | low_cardinality     | 3                   

In [ ]:
for col in eda['categorical_summary']:
    sample = sample_dataframe(eda['categorical_summary'][col], head=1, random=3, tail=1)
    sample_df = pd.concat(
        [sample[k] for k in ['head', 'random', 'tail']],
        ignore_index=False
    )
    print(sample_df.to_markdown())
    print()

|                          |   count |   percent |   Survived_0_% |   Survived_1_% |
|:-------------------------|--------:|----------:|---------------:|---------------:|
| Dooley, Mr. Patrick      |       1 |      0.11 |            100 |              0 |
| Braund, Mr. Owen Harris  |       1 |      0.11 |            100 |              0 |
| Masselmani, Mrs. Fatima  |       1 |      0.11 |              0 |            100 |
| Moran, Mr. James         |       1 |      0.11 |            100 |              0 |
| Bonnell, Miss. Elizabeth |       1 |      0.11 |              0 |            100 |

| Sex    |   count |   percent |   Survived_0_% |   Survived_1_% |
|:-------|--------:|----------:|---------------:|---------------:|
| male   |     577 |     64.76 |          81.11 |          18.89 |
| female |     314 |     35.24 |          25.8  |          74.2  |

|                  |   count |   percent |   Survived_0_% |   Survived_1_% |
|:-----------------|--------:|----------:|---------------:

In [ ]:
for col in eda['numerical_summary']:
    print(f"Numerical column: {col}")
    print(eda['numerical_summary'][col].to_markdown())
    print()

Numerical column: PassengerId
|                         | 0                   |
|:------------------------|:--------------------|
| count                   | 891.0               |
| mean                    | 446.0               |
| std                     | 257.3538420152301   |
| min                     | 1.0                 |
| 25%                     | 223.5               |
| 50%                     | 446.0               |
| 75%                     | 668.5               |
| max                     | 891.0               |
| missing_count           | 0                   |
| missing_%               | 0.0                 |
| outlier_count           | 0                   |
| outlier_%               | 0.0                 |
| skew                    | 0.0                 |
| skew_classification     | low skew            |
| kurtosis                | -1.1999999999999997 |
| kurtosis_classification | normal_tails        |

Numerical column: Pclass
|                         | 0               

In [ ]:
print("Correlation matrix:")
print(eda["correlation_matrix"].to_markdown())

Correlation matrix:
|             |   PassengerId |   Survived |   Pclass |    Age |   SibSp |   Parch |   Fare |
|:------------|--------------:|-----------:|---------:|-------:|--------:|--------:|-------:|
| PassengerId |         1     |     -0.005 |   -0.035 |  0.037 |  -0.058 |  -0.002 |  0.013 |
| Survived    |        -0.005 |      1     |   -0.338 | -0.077 |  -0.035 |   0.082 |  0.257 |
| Pclass      |        -0.035 |     -0.338 |    1     | -0.369 |   0.083 |   0.018 | -0.549 |
| Age         |         0.037 |     -0.077 |   -0.369 |  1     |  -0.308 |  -0.189 |  0.096 |
| SibSp       |        -0.058 |     -0.035 |    0.083 | -0.308 |   1     |   0.415 |  0.16  |
| Parch       |        -0.002 |      0.082 |    0.018 | -0.189 |   0.415 |   1     |  0.216 |
| Fare        |         0.013 |      0.257 |   -0.549 |  0.096 |   0.16  |   0.216 |  1     |


In [ ]:
print('Correlation with the target variable:')
print(eda["target_correlation"].to_markdown())

Correlation with the target variable:
|             |   Survived |
|:------------|-----------:|
| Pclass      |     -0.338 |
| Fare        |      0.257 |
| Parch       |      0.082 |
| Age         |     -0.077 |
| SibSp       |     -0.035 |
| PassengerId |     -0.005 |


In [ ]:
for col in eda["categorical_rare"]:
    print(f"Categorical column with rare values: {col}")
    print(eda["categorical_rare"][col])
    print()

Categorical column with rare values: Name
No rare categories - too many unique values

Categorical column with rare values: Sex
No rare categories

Categorical column with rare values: Ticket
No rare categories - too many unique values

Categorical column with rare values: Cabin
No rare categories - too many unique values

Categorical column with rare values: Embarked
          count  percent
Embarked                
MISSING       2     0.22



In [ ]:
# print(eda['sample'])
sample_df = pd.concat(
    [eda['sample'][k] for k in ['head', 'random', 'tail']],
    ignore_index=True
)
print(sample_df.to_markdown(index=False))

|   PassengerId |   Survived |   Pclass | Name                                                | Sex    |   Age |   SibSp |   Parch | Ticket           |    Fare | Cabin   | Embarked   |
|--------------:|-----------:|---------:|:----------------------------------------------------|:-------|------:|--------:|--------:|:-----------------|--------:|:--------|:-----------|
|             1 |          0 |        3 | Braund, Mr. Owen Harris                             | male   |    22 |       1 |       0 | A/5 21171        |  7.25   | nan     | S          |
|             2 |          1 |        1 | Cumings, Mrs. John Bradley (Florence Briggs Thayer) | female |    38 |       1 |       0 | PC 17599         | 71.2833 | C85     | C          |
|             3 |          1 |        3 | Heikkinen, Miss. Laina                              | female |    26 |       0 |       0 | STON/O2. 3101282 |  7.925  | nan     | S          |
|            48 |          1 |        3 | O'Driscoll, Miss. Bridget        